# Lecture 6: Loading Models with `from_pretrained`
In this notebook, we will explore how to load pre-trained models using the `from_pretrained` method from the Hugging Face Transformers library. We will also dive into the configuration, weights, and caching mechanisms.

The Google Colab versin of this notebook is available here: https://colab.research.google.com/drive/1gb3hu83Wktk5cUDObPJqjlM5olAhbvam?usp=sharing


## **Feeling Brave?**

Try out the code for this lecture on your laptop or desktop. The same code in the above mentioned Google Colab is also available here for anyone who feels they want to take on the challenge of running this lecture on their machine.


**Consider the following** before running this notebook on your computer:
1. Make sure you have plenty of RAM (ideally >= 16 GB)

2. If you **do not have** an NVIDIA GPU, you will have to install bitsandbytes cour version by following these steps
    - In your terminal, activate your virtual environment and run `pip uninstall bitsandbytes` or in your notebook cell run `!pip uninstall bitsandbytes`
    - In your terminal run `pip install bitsandbytes-cpu` or in your notebook cell run `!pip install bitsandbytes-cpu`

3. If you **do** have an NVIDIA GPU
    - In your terminal, activate your virtual environment and run `pip uninstall torch torchvision torchaudio` or in your notebook cell run `!pip uninstall torch torchvision torchaudio`
    - In your terminal run `pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129` or in your notebook cell run `!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129`

4. Make sure you have a stable internet connection as the download can take some time

*Note* that running models without a GPU can be extremely time consuming and may lead to your machine overheating if it is old.

# Step 1: Load libraries and log in to Huggingface

In [11]:
import os

import torch
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()

hf_token=os.getenv("HUGGINFACE_API_KEY")

login(hf_token)

if hf_token is None:
    raise Exception("HF API key is missing")

# Step 2: Load quantization configuration for model

In [ ]:
# My computer has enough RAM and no cuda so I tried the model as is, i.e. without quantizing
# I couldn't find the cpu version of bitsandbytes


# from transformers import BitsAndBytesConfig

# Quantization Config - this allows us to load the model into memory and use less memory
# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_quant_type="nf4"
# )

# This is a CPU equivalent of the GPU quantization config
# Uncomment the below if you are only using CPU

# quant_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=float32,
#     bnb_4bit_quant_type="nf4"
# )

# Step 3: Load a Pre-trained Model
We will use the `meta-llama/Meta-Llama-3.1-8B-Instruct` model as an example. This step demonstrates how to load the model and tokenizer.

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# The model name changed. Updated model?

model_name = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name) # no quant config needed/used
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

# from transformers import AutoModelForCausalLM

# # model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# model_name = "meta-llama/Llama-3.1-8B-Instruct"

# # model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", quantization_config=quant_config)
# model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

print(f"Model '{model_name}' loaded successfully!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>
Model 'meta-llama/Llama-3.1-8B-Instruct' loaded successfully!


# Step 4: Explore the Model Configuration
The configuration of a model contains important details such as the number of layers, hidden size, and more.

In [4]:
config = model.config
print("Model Configuration:")
print(config)

Model Configuration:
LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.8.1",
  "use_cache": true,
  "vocab_size": 128256
}



Let's have a look at the actual layers (just for fun!)

In [5]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

# Step 5: Understand Caching
When you load a model, it is cached locally to avoid downloading it again. The models are usually stored in the following path: ~/.cache/huggingface/hub by default

See further reference: [Huggingface cache management](~/.cache/huggingface/hub)

# Step 6: Tokenizing a Prompt and Generating Text
In this step, we will tokenize a list of messages, pass it to the model, and generate text as output.

In [6]:
# An instruct model requires a list of messages as we saw in AI Engineering Essentials Part 1
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is the best way to structure and organize my thoughts?"}
  ]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt") # removed to cuda

print(inputs)

{'input_ids': tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271,   2675,    527,
            264,  11190,  18328, 128009, 128006,    882, 128007,    271,   3923,
            374,    279,   1888,   1648,    311,   6070,    323,  31335,    856,
          11555,     30, 128009]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
# Pass the input IDs to the model to generate output
# output_ids = model.generate(inputs, max_new_tokens=80)
output_ids = model.generate(**inputs, max_new_tokens=80)

# Display the generated output IDs
print("Generated Output IDs:", output_ids)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generated Output IDs: tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1627,  10263,    220,   2366,     19,    271,   2675,    527,
            264,  11190,  18328, 128009, 128006,    882, 128007,    271,   3923,
            374,    279,   1888,   1648,    311,   6070,    323,  31335,    856,
          11555,     30, 128009, 128006,  78191, 128007,    271,   3947,    527,
           3892,   7524,   5627,    311,   6070,    323,  31335,    701,  11555,
             11,    323,    279,   1888,   1749,    369,    499,    690,   6904,
            389,    701,   4443,  19882,    323,   6975,   1742,     13,   5810,
            527,   1063,   5526,  12823,   1473,     16,     13,   3146,  70738,
          39546,  96618,    362,   9302,   5507,    430,   5829,  26432,     11,
          38057,     11,    323,  21513,    311,   1893,    264,  13861,    315,
      

In [9]:
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)

print("Generated Text:", generated_text)

Generated Text: ['system\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistantuser\n\nWhat is the best way to structure and organize my thoughts?assistant\n\nThere are several effective ways to structure and organize your thoughts, and the best method for you will depend on your personal preferences and learning style. Here are some popular techniques:\n\n1. **Mind Mapping**: A visual tool that uses circles, arrows, and keywords to create a diagram of your thoughts. Start with a central idea and branch out to related concepts.\n2. **Outline Method']


# Takeaways
- The `from_pretrained` method simplifies loading pre-trained models and tokenizers.
- Models are cached locally for efficiency.
-- I moved the higginface model cache to an external drive :-)
- You can explore model configurations and map models to devices for optimized inference.

# Your Challenge
Fork this notebook, change the model ID to one in your native language, and share your results in the course repository!